In [1]:
import random
import torchaudio
import torch
from transformers import AutoConfig, AutoModel, WhisperProcessor
from tqdm import tqdm
import pandas as pd
from sklearn.neural_network import MLPClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device available is', device)

seed = 7 
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# TOGGLE THIS ON IF YOU WANT TO RE-GENERATE THE EMBEDDING SETS
generate = True

C:\Users\dunna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device available is cuda


## Get our encoders ready

In [2]:
# wav2vec embedding method

wav2vec_config = AutoConfig.from_pretrained("facebook/wav2vec2-base")
wav2vec = AutoModel.from_pretrained("facebook/wav2vec2-base", device_map="cuda")

def generate_wav2vec_embedding(waveform):
  # extract features
  emission = wav2vec(waveform.to(device)).last_hidden_state
  emission = emission.detach().cpu()
  # emission = emission.detach().cpu().numpy()
  return emission

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 2180.38it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# whisper embedding method
whisper_config = AutoConfig.from_pretrained("openai/whisper-large-v3-turbo")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")
whisper = AutoModel.from_pretrained("openai/whisper-large-v3-turbo", device_map="cuda")

def generate_whisper_embedding(input_features):

  # no clue why this is required but it was in the docs and throws and error if not included
  decoder_input_ids = torch.tensor([[1, 1]]) * whisper.config.decoder_start_token_id

  emission = whisper(input_features.to(device), decoder_input_ids=decoder_input_ids.to(device)).last_hidden_state
  emission = emission.detach().cpu()
  # emission = emission.detach().cpu().numpy()
  return emission

Loading weights: 100%|██████████| 587/587 [00:00<00:00, 1624.24it/s]


## Set up timit data
First, we set up our timit data so that we have phone-audio pairings to train over.
`.phn` files in the corpus have triples of (beginning sample, ending sample, phone).

In [4]:
# get data about train/test split
test_df = pd.read_csv('data/darpa-timit-acousticphonetic-continuous-speech/test_data.csv')
test_df = test_df[test_df['is_audio'] == True]
test_df = test_df[test_df['filename'].str.contains('.wav')] # for some reason, the wav files have both .WAV and .WAV.wav files. Only the .WAV.wav files are readable, so we filter the others out

train_df = pd.read_csv('data/darpa-timit-acousticphonetic-continuous-speech/train_data.csv')
train_df = train_df[train_df['is_audio'] == True]
train_df = train_df[train_df['filename'].str.contains('.wav')] # for some reason, the wav files have both .WAV and .WAV.wav files. Only the .WAV.wav files are readable, so we filter the others out

# helper functions

def get_phn_file(audio_path):
    part = audio_path.split('.')[0] # the part before the extension
    return part + '.PHN'



In [5]:
# embedding helpers

def get_wav2vec_embedding(audio_path):
    # returns a list of embeddings and a list with the corresponding phone labels
    phone_path = get_phn_file(audio_path)
    loaded = []
    labels = []
    
    with open(phone_path) as file:
        for line in file:
            begin, end, phone = line.split()
            begin = int(begin)
            end = int(end)

            if phone == '#h' or phone == 'h#': # file boundary
                pass

            else:
                num_frames = end - begin
                if num_frames > 400: # otherwise wav2vec freaks out

                    audio, sample_rate = torchaudio.load(audio_path, frame_offset=begin, num_frames=num_frames)

                    with torch.no_grad():
                        embedding = generate_wav2vec_embedding(audio)
                    embedding = torch.mean(embedding, dim=1) # average over all frames the phone is present for
                    
                    shape = embedding.shape
                    if shape[1] == 768:
                        embedding = embedding.squeeze()
                        loaded.append(embedding)
                        labels.append(phone)
    
    return loaded, labels

def get_whisper_embedding(audio_path):
    # returns a list of embeddings and a list with the corresponding phone labels
    phone_path = get_phn_file(audio_path)
    loaded = []
    labels = []
    
    with open(phone_path) as file:
        for line in file:
            begin, end, phone = line.split()
            begin = int(begin)
            end = int(end)

            if phone in ['h#', 'epi', 'pau', '1', '2']: # non-phone info
                pass

            else:
                num_frames = end - begin
                if num_frames > 400: # otherwise wav2vec freaks out

                    audio, sample_rate = torchaudio.load(audio_path, frame_offset=begin, num_frames=num_frames)

                    audio = whisper_processor(
                        audio.squeeze().numpy(),
                        sampling_rate=16000,
                        return_tensors="pt"
                    )

                    input_features = audio["input_features"].to(dtype=torch.float16).to(device) # otherwise whisper freaks out

                    with torch.no_grad():
                        embedding = generate_whisper_embedding(input_features)
                    embedding = torch.mean(embedding, dim=1) # average over all frames the phone is present for
                    
                    shape = embedding.shape
                    # print(shape)
                    if shape[1] == 1280:
                        embedding = embedding.squeeze()
                        loaded.append(embedding)
                        labels.append(phone)
    
    return loaded, labels

In [6]:
# generate our whisper embeddings

if generate:
    whisper_train_embeddings = []
    whisper_train_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(train_df.itertuples(), "Generating train whisper embeddings", total=train_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_whisper_embedding(audio_path)

        whisper_train_embeddings.extend(x)
        whisper_train_labels.extend(y)

    # save to disk
    torch.save(whisper_train_embeddings, 'data/saved_embeddings/whisper_train_embeddings.pt')
    torch.save(whisper_train_labels, 'data/saved_embeddings/whisper_train_labels.pt')

Generating train whisper embeddings: 100%|██████████| 4620/4620 [2:36:57<00:00,  2.04s/it]  


In [ ]:
if generate:
    whisper_test_embeddings = []
    whisper_test_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(test_df.itertuples(), "Generating test whisper embeddings", total=test_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_whisper_embedding(audio_path)

        whisper_test_embeddings.extend(x)
        whisper_test_labels.extend(y)

    # save to disk
    torch.save(whisper_test_embeddings, 'data/saved_embeddings/whisper_test_embeddings.pt')
    torch.save(whisper_test_labels, 'data/saved_embeddings/whisper_test_labels.pt')

Generating test whisper embeddings:   0%|          | 0/1680 [00:02<?, ?it/s]


NameError: name 'whisper_train_embeddings' is not defined

In [ ]:
# generate our wav2vec train embeddings

if generate:
    wav2vec_train_embeddings = []
    wav2vec_train_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(train_df.itertuples(), "Generating train wav2vec embeddings", total=train_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_wav2vec_embedding(audio_path)

        wav2vec_train_embeddings.extend(x)
        wav2vec_train_labels.extend(y)

    # save to disk
    torch.save(wav2vec_train_embeddings, 'data/saved_embeddings/wav2vec_train_embeddings.pt')
    torch.save(wav2vec_train_labels, 'data/saved_embeddings/wav2vec_train_labels.pt')

Generating train wav2vec embeddings: 100%|██████████| 4620/4620 [19:43<00:00,  3.90it/s]


In [ ]:
# empty previous variables
wav2vec_train_embeddings = None
wav2vec_train_labels = None

In [ ]:
# generate our wav2vec test embeddings

if generate:
    wav2vec_test_embeddings = []
    wav2vec_test_labels = []

    path_to_timit = 'data/darpa-timit-acousticphonetic-continuous-speech/data/'

    for row in tqdm(test_df.itertuples(), "Generating test wav2vec embeddings", total=test_df['index'].count()):
        # print(row)
        audio_path = path_to_timit + row[6] # [6] is the "path from data source" one

        x, y = get_wav2vec_embedding(audio_path)

        wav2vec_test_embeddings.extend(x)
        wav2vec_test_labels.extend(y)

    # save to disk
    torch.save(wav2vec_test_embeddings, 'data/saved_embeddings/wav2vec_test_embeddings.pt')
    torch.save(wav2vec_test_labels, 'data/saved_embeddings/wav2vec_test_labels.pt')

Generating test wav2vec embeddings: 100%|██████████| 1680/1680 [07:10<00:00,  3.91it/s]
